# Panoptic Segmentation Mechanics & Panoptic Quality (PQ)

## 1. Unifying "Stuff" and "Things"
Computer vision traditionally divided pixel-level understanding into two isolated tasks:
- **Semantic Segmentation**: Dense classification of non-countable amorphous regions (**Stuff**: sky, road, grass, water). All pixels of the road share one label; individual lanes are not distinguished.
- **Instance Segmentation**: Detecting and segmenting countable objects (**Things**: person, bicycle, car). Multiple pedestrians get distinct instance IDs, but the background sky is ignored.

### Panoptic Segmentation Formulation (Kirillov et al., CVPR 2019)
Panoptic segmentation requires every pixel to be assigned a **semantic class** and an **instance ID**:
- If pixel $p$ is stuff: $\text{Semantic Label} = \text{Sky}$, $\text{Instance ID} = 0$.
- If pixel $p$ is thing: $\text{Semantic Label} = \text{Person}$, $\text{Instance ID} = 3$.

### Panoptic Quality (PQ) Metric
Panoptic Quality evaluates performance without favoring either things or stuff:

$$\text{PQ} = \underbrace{\frac{\sum_{(p, g) \in \text{TP}} \operatorname{IoU}(p, g)}{|\text{TP}|}}_{\text{Segmentation Quality (SQ)}} \times \underbrace{\frac{|\text{TP}|}{|\text{TP}| + \frac{1}{2}|\text{FP}| + \frac{1}{2}|\text{FN}|}}_{\text{Recognition Quality (RQ)}}$$

Where:
- Predicted segment $p$ matches ground truth segment $g$ as a **True Positive (TP)** if and only if $\operatorname{IoU}(p, g) > 0.5$.
- Because IoU $> 0.5$, matching is mathematically unique (each ground truth can match at most one prediction).

In [ ]:
import numpy as np

def compute_panoptic_quality(ground_truth_segments, predicted_segments, iou_threshold=0.5):
    """
    Evaluates Panoptic Quality (PQ), Segmentation Quality (SQ), and Recognition Quality (RQ).
    Each segment is a binary mask [H, W].
    """
    TP = []
    FP = []
    matched_gt = set()
    
    for p_idx, p_mask in enumerate(predicted_segments):
        best_iou = 0.0
        best_g_idx = -1
        
        for g_idx, g_mask in enumerate(ground_truth_segments):
            if g_idx in matched_gt:
                continue
            intersection = np.logical_and(p_mask, g_mask).sum()
            union = np.logical_or(p_mask, g_mask).sum()
            iou = intersection / max(union, 1)
            if iou > best_iou:
                best_iou = iou
                best_g_idx = g_idx
                
        if best_iou > iou_threshold:
            TP.append(best_iou)
            matched_gt.add(best_g_idx)
        else:
            FP.append(p_idx)
            
    FN = [g_idx for g_idx in range(len(ground_truth_segments)) if g_idx not in matched_gt]
    
    # Calculate SQ, RQ, PQ
    n_tp = len(TP)
    n_fp = len(FP)
    n_fn = len(FN)
    
    sq = np.mean(TP) if n_tp > 0 else 0.0
    rq = n_tp / (n_tp + 0.5 * n_fp + 0.5 * n_fn)
    pq = sq * rq
    
    return {"PQ": pq, "SQ": sq, "RQ": rq, "TP": n_tp, "FP": n_fp, "FN": n_fn}

# Demonstration
H, W = 50, 50
gt1 = np.zeros((H, W), dtype=bool); gt1[10:30, 10:30] = True # Person 1
gt2 = np.zeros((H, W), dtype=bool); gt2[35:45, 35:45] = True # Person 2

# Pred 1 is slightly offset from gt1 (IoU ~ 0.70)
pred1 = np.zeros((H, W), dtype=bool); pred1[12:32, 12:32] = True
# Pred 2 matches gt2 perfectly
pred2 = np.copy(gt2)
# Pred 3 is a hallucinated false positive
pred3 = np.zeros((H, W), dtype=bool); pred3[0:10, 0:10] = True

res = compute_panoptic_quality([gt1, gt2], [pred1, pred2, pred3])
print(f"Panoptic Evaluation Metrics:")
print(f"  Segmentation Quality (SQ) : {res['SQ']:.3f} (Mean IoU of True Positives)")
print(f"  Recognition Quality  (RQ) : {res['RQ']:.3f} (F1-style detection score)")
print(f"  Panoptic Quality     (PQ) : {res['PQ']:.3f} (Unified metric SQ * RQ)")
print(f"  Counts: TP={res['TP']}, FP={res['FP']}, FN={res['FN']}")

## 2. Why IoU Threshold $> 0.5$ Guarantees Unique Bipartite Matching
If two distinct predicted segments $p_1$ and $p_2$ both overlapped with ground truth segment $g$ with $\operatorname{IoU} > 0.5$:

$$\operatorname{IoU}(p_1, g) + \operatorname{IoU}(p_2, g) > 1.0$$

Since predicted segments are non-overlapping in panoptic segmentation ($p_1 \cap p_2 = \emptyset$), their joint union would exceed the area of $g$, which is mathematically impossible! Hence, **no complex Hungarian bipartite matching algorithm is needed** for evaluating Panoptic Quality when threshold $\ge 0.5$.

## 3. High-Yield Interview Questions & Best Practices

### Q1: How do modern transformers like Mask2Former unify Semantic, Instance, and Panoptic Segmentation?
**Answer:** Mask2Former uses a single universal architecture based on **masked-attention query transformers**:
1. It maintains $N$ learnable object queries.
2. Each query predicts a binary mask $m_i \in [0, 1]^{H \times W}$ and a class probability distribution $p_i$.
3. To produce panoptic predictions: query masks are filtered by score and merged greedily into a non-overlapping panoptic map. If a query corresponds to a "stuff" class, overlapping segments are merged; if "thing", each query retains an independent instance ID.

### Q2: What happens when two instances overlap in Panoptic Segmentation?
**Answer:** By definition, panoptic segmentation enforces **disjoint pixel partitions**: no pixel can belong to two instances simultaneously. If an occlusion occurs (e.g., a person standing in front of a car), the model must assign the occluded pixels strictly to the foreground object (the person). Instance segmentation allows overlapping bounding masks, but panoptic segmentation requires resolving depths and occlusions.